## Imports & Paths

In [1]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import zoom

# ── Paths (same dataset, new results folder) ───────────────
DATASET_PATH  = Path(r"C:\Users\Admin\Desktop\madhura\Task02_Heart\Task02_Heart")
IMAGES_TR     = DATASET_PATH / "imagesTr"
LABELS_TR     = DATASET_PATH / "labelsTr"
IMAGES_TS     = DATASET_PATH / "imagesTs"

RESULTS_PATH  = Path(r"C:\Users\Admin\Desktop\madhura\nnunet_project\results\02_preprocessing")
PREPROCESSED  = Path(r"C:\Users\Admin\Desktop\madhura\nnunet_project\preprocessed")

# Subfolders for preprocessed output
PREP_IMGS_TR  = PREPROCESSED / "imagesTr"
PREP_LBLS_TR  = PREPROCESSED / "labelsTr"
PREP_IMGS_TS  = PREPROCESSED / "imagesTs"

# Create all folders
for folder in [RESULTS_PATH, PREP_IMGS_TR, PREP_LBLS_TR, PREP_IMGS_TS]:
    folder.mkdir(parents=True, exist_ok=True)

# Valid case IDs from notebook 1
VALID_CASES = [
    'la_003','la_004','la_005','la_007','la_009','la_010',
    'la_011','la_014','la_016','la_017','la_018','la_019',
    'la_020','la_021','la_022','la_023','la_024','la_026',
    'la_029','la_030'
]

print("✅ All folders created")
print(f"   Dataset     : {DATASET_PATH}")
print(f"   Results     : {RESULTS_PATH}")
print(f"   Preprocessed: {PREPROCESSED}")

✅ All folders created
   Dataset     : C:\Users\Admin\Desktop\madhura\Task02_Heart\Task02_Heart
   Results     : C:\Users\Admin\Desktop\madhura\nnunet_project\results\02_preprocessing
   Preprocessed: C:\Users\Admin\Desktop\madhura\nnunet_project\preprocessed


## Define Target Spacing & Compute Resampled Shapes

In [2]:
# ── Preprocessing parameters ───────────────────────────────
# From notebook 1: all cases have spacing (1.25, 1.25, 1.37)
# We resample to isotropic 1.25mm (no X/Y change, upsample Z)
ORIGINAL_SPACING = (1.25, 1.25, 1.37)
TARGET_SPACING   = (1.25, 1.25, 1.25)  # isotropic

print("=" * 55)
print("RESAMPLING PLAN")
print("=" * 55)
print(f"Original spacing : {ORIGINAL_SPACING} mm")
print(f"Target spacing   : {TARGET_SPACING} mm")
print(f"Zoom factor Z    : {ORIGINAL_SPACING[2] / TARGET_SPACING[2]:.4f}")

print("\nPer-case shape after resampling:")
print(f"{'Case':<12} {'Original Shape':<22} {'Resampled Shape'}")
print("-" * 55)

shapes_after = []
for case_id in VALID_CASES:
    img = nib.load(IMAGES_TR / f"{case_id}.nii.gz")
    orig_shape = img.shape
    zoom_factors = tuple(
        ORIGINAL_SPACING[i] / TARGET_SPACING[i] for i in range(3)
    )
    new_shape = tuple(
        int(round(orig_shape[i] * zoom_factors[i])) for i in range(3)
    )
    shapes_after.append(new_shape)
    print(f"{case_id:<12} {str(orig_shape):<22} {str(new_shape)}")

shapes_after = np.array(shapes_after)
print("\n" + "=" * 55)
print("RESAMPLED SHAPE STATS")
print("=" * 55)
print(f"X range : {shapes_after[:,0].min()} – {shapes_after[:,0].max()}")
print(f"Y range : {shapes_after[:,1].min()} – {shapes_after[:,1].max()}")
print(f"Z range : {shapes_after[:,2].min()} – {shapes_after[:,2].max()}")
print(f"\nMax shape across all cases: {shapes_after.max(axis=0)}")
print(f"→ Padding target will be  : {shapes_after.max(axis=0)}")

RESAMPLING PLAN
Original spacing : (1.25, 1.25, 1.37) mm
Target spacing   : (1.25, 1.25, 1.25) mm
Zoom factor Z    : 1.0960

Per-case shape after resampling:
Case         Original Shape         Resampled Shape
-------------------------------------------------------
la_003       (320, 320, 130)        (320, 320, 142)
la_004       (320, 320, 110)        (320, 320, 121)
la_005       (320, 320, 120)        (320, 320, 132)
la_007       (320, 320, 130)        (320, 320, 142)
la_009       (320, 320, 100)        (320, 320, 110)
la_010       (320, 320, 120)        (320, 320, 132)
la_011       (320, 320, 120)        (320, 320, 132)
la_014       (320, 320, 120)        (320, 320, 132)
la_016       (320, 320, 90)         (320, 320, 99)
la_017       (320, 320, 120)        (320, 320, 132)
la_018       (320, 320, 122)        (320, 320, 134)
la_019       (320, 320, 100)        (320, 320, 110)
la_020       (320, 320, 110)        (320, 320, 121)
la_021       (320, 320, 100)        (320, 320, 110)
la_022 

## Define Preprocessing Functions

In [3]:
def resample_volume(data, original_spacing, target_spacing, is_label=False):
    """Resample volume to target spacing using zoom."""
    zoom_factors = tuple(
        original_spacing[i] / target_spacing[i] for i in range(3)
    )
    if is_label:
        # Nearest neighbour for labels to preserve binary values
        resampled = zoom(data, zoom_factors, order=0)
    else:
        # Linear interpolation for images
        resampled = zoom(data, zoom_factors, order=1)
    return resampled


def pad_to_shape(data, target_shape, pad_value=0):
    """Pad volume symmetrically to target shape."""
    pad_width = []
    for i in range(3):
        diff = target_shape[i] - data.shape[i]
        pad_before = diff // 2
        pad_after  = diff - pad_before
        pad_width.append((pad_before, pad_after))
    return np.pad(data, pad_width, mode="constant", constant_values=pad_value)


def zscore_normalise(data, mask=None):
    """Z-score normalisation using foreground voxels only."""
    if mask is not None:
        fg = data[mask > 0]
    else:
        # Use non-zero voxels as foreground
        fg = data[data > 0]
    mean = fg.mean()
    std  = fg.std()
    normalised = (data - mean) / (std + 1e-8)
    return normalised, mean, std


print("✅ Functions defined:")
print("   resample_volume()  — isotropic resampling")
print("   pad_to_shape()     — symmetric zero padding")
print("   zscore_normalise() — foreground z-score norm")

# Target shape for all volumes
TARGET_SHAPE = (320, 320, 142)
print(f"\n✅ Target shape set: {TARGET_SHAPE}")

✅ Functions defined:
   resample_volume()  — isotropic resampling
   pad_to_shape()     — symmetric zero padding
   zscore_normalise() — foreground z-score norm

✅ Target shape set: (320, 320, 142)


## Preprocess All Training Cases

In [4]:
import time

print("=" * 55)
print("PREPROCESSING TRAINING CASES")
print("=" * 55)

norm_stats = []  # store per-case normalisation stats
start_total = time.time()

for idx, case_id in enumerate(VALID_CASES):
    start = time.time()

    # ── 1. Load ───────────────────────────────────────────
    img_nib = nib.load(IMAGES_TR / f"{case_id}.nii.gz")
    lbl_nib = nib.load(LABELS_TR / f"{case_id}.nii.gz")
    img     = img_nib.get_fdata()
    lbl     = lbl_nib.get_fdata()
    orig_spacing = img_nib.header.get_zooms()[:3]

    # ── 2. Resample ───────────────────────────────────────
    img_res = resample_volume(img, orig_spacing, TARGET_SPACING, is_label=False)
    lbl_res = resample_volume(lbl, orig_spacing, TARGET_SPACING, is_label=True)

    # ── 3. Pad ────────────────────────────────────────────
    img_pad = pad_to_shape(img_res, TARGET_SHAPE, pad_value=0)
    lbl_pad = pad_to_shape(lbl_res, TARGET_SHAPE, pad_value=0)

    # ── 4. Normalise ──────────────────────────────────────
    img_norm, mean, std = zscore_normalise(img_pad)

    # ── 5. Verify label integrity ─────────────────────────
    unique_vals = np.unique(lbl_pad)

    # ── 6. Save as .npy ───────────────────────────────────
    np.save(PREP_IMGS_TR / f"{case_id}.npy", img_norm.astype(np.float32))
    np.save(PREP_LBLS_TR / f"{case_id}.npy", lbl_pad.astype(np.uint8))

    elapsed = time.time() - start
    norm_stats.append({
        "case_id": case_id,
        "shape_after": img_norm.shape,
        "mean": round(mean, 3),
        "std": round(std, 3),
        "label_unique": unique_vals,
        "atrium_%": round(100 * (lbl_pad == 1).sum() / lbl_pad.size, 4)
    })

    print(f"[{idx+1:02d}/20] {case_id} | "
          f"shape={img_norm.shape} | "
          f"mean={mean:.1f} std={std:.1f} | "
          f"labels={unique_vals} | "
          f"{elapsed:.1f}s")

total_time = time.time() - start_total
print(f"\n✅ All 20 cases preprocessed in {total_time:.1f}s")
print(f"   Saved to: {PREP_IMGS_TR}")

PREPROCESSING TRAINING CASES
[01/20] la_003 | shape=(320, 320, 142) | mean=260.2 std=285.0 | labels=[0. 1.] | 1.6s
[02/20] la_004 | shape=(320, 320, 142) | mean=204.8 std=255.8 | labels=[0. 1.] | 1.3s
[03/20] la_005 | shape=(320, 320, 142) | mean=233.9 std=253.0 | labels=[0. 1.] | 1.4s
[04/20] la_007 | shape=(320, 320, 142) | mean=262.6 std=248.6 | labels=[0. 1.] | 1.5s
[05/20] la_009 | shape=(320, 320, 142) | mean=247.6 std=287.9 | labels=[0. 1.] | 1.2s
[06/20] la_010 | shape=(320, 320, 142) | mean=243.5 std=276.2 | labels=[0. 1.] | 1.4s
[07/20] la_011 | shape=(320, 320, 142) | mean=295.7 std=297.1 | labels=[0. 1.] | 1.4s
[08/20] la_014 | shape=(320, 320, 142) | mean=284.2 std=283.3 | labels=[0. 1.] | 1.4s
[09/20] la_016 | shape=(320, 320, 142) | mean=292.3 std=300.4 | labels=[0. 1.] | 1.1s
[10/20] la_017 | shape=(320, 320, 142) | mean=271.1 std=280.1 | labels=[0. 1.] | 1.4s
[11/20] la_018 | shape=(320, 320, 142) | mean=221.3 std=255.4 | labels=[0. 1.] | 1.4s
[12/20] la_019 | shape=(3

## Check Test Case Shapes First

In [6]:
print("=" * 55)
print("TEST CASE SHAPES")
print("=" * 55)

test_cases = sorted([
    f.name.replace(".nii.gz", "") 
    for f in IMAGES_TS.iterdir() 
    if f.suffix == ".gz" and not f.name.startswith("._")
])

max_z = 0
for case_id in test_cases:
    img_nib      = nib.load(IMAGES_TS / f"{case_id}.nii.gz")
    orig_shape   = img_nib.shape
    orig_spacing = img_nib.header.get_zooms()[:3]
    zoom_factors = tuple(
        float(orig_spacing[i]) / TARGET_SPACING[i] for i in range(3)
    )
    new_shape = tuple(
        int(round(orig_shape[i] * zoom_factors[i])) for i in range(3)
    )
    max_z = max(max_z, new_shape[2])
    print(f"{case_id} | orig={orig_shape} | resampled={new_shape}")

print(f"\nMax Z across test cases : {max_z}")
print(f"Max Z across train cases: 142")
print(f"\n→ New TARGET_SHAPE needed: (320, 320, {max(142, max_z)})")

TEST CASE SHAPES
la_001 | orig=(400, 400, 180) | resampled=(240, 240, 122)
la_002 | orig=(320, 320, 140) | resampled=(320, 320, 153)
la_006 | orig=(400, 400, 180) | resampled=(240, 240, 108)
la_008 | orig=(320, 320, 110) | resampled=(320, 320, 121)
la_012 | orig=(320, 320, 137) | resampled=(320, 320, 150)
la_013 | orig=(320, 320, 120) | resampled=(320, 320, 132)
la_015 | orig=(320, 320, 100) | resampled=(320, 320, 110)
la_025 | orig=(320, 320, 120) | resampled=(320, 320, 132)
la_027 | orig=(320, 320, 100) | resampled=(320, 320, 110)
la_028 | orig=(320, 320, 110) | resampled=(320, 320, 121)

Max Z across test cases : 153
Max Z across train cases: 142

→ New TARGET_SHAPE needed: (320, 320, 153)


## Update Target Shape & Fix Padding/Cropping Function

In [7]:
# Updated target shape to fit all train + test cases
TARGET_SHAPE = (320, 320, 153)
print(f"✅ Updated TARGET_SHAPE: {TARGET_SHAPE}")

def pad_or_crop_to_shape(data, target_shape, pad_value=0):
    """Pad OR crop each dimension to exactly target_shape."""
    result = data.copy()
    for i in range(3):
        diff = target_shape[i] - result.shape[i]
        if diff > 0:
            # Need to pad
            pad_before = diff // 2
            pad_after  = diff - pad_before
            pad_width  = [(0,0)] * 3
            pad_width[i] = (pad_before, pad_after)
            result = np.pad(result, pad_width, 
                           mode="constant", constant_values=pad_value)
        elif diff < 0:
            # Need to crop (center crop)
            crop_before = abs(diff) // 2
            crop_after  = result.shape[i] - (abs(diff) - crop_before)
            slices = [slice(None)] * 3
            slices[i] = slice(crop_before, crop_after)
            result = result[tuple(slices)]
    return result

print("✅ pad_or_crop_to_shape() defined")
print("   → Handles both padding AND cropping per dimension")

# Quick test
dummy = np.zeros((240, 240, 122))
out   = pad_or_crop_to_shape(dummy, TARGET_SHAPE)
print(f"\n✅ Test: (240,240,122) → {out.shape} ✓" 
      if out.shape == TARGET_SHAPE else f"❌ Test failed: {out.shape}")

✅ Updated TARGET_SHAPE: (320, 320, 153)
✅ pad_or_crop_to_shape() defined
   → Handles both padding AND cropping per dimension

✅ Test: (240,240,122) → (320, 320, 153) ✓


## Re-Preprocess All Training Cases with New Target Shape

In [8]:
print("=" * 55)
print("RE-PREPROCESSING TRAINING CASES (320,320,153)")
print("=" * 55)

norm_stats = []
start_total = time.time()

for idx, case_id in enumerate(VALID_CASES):
    start = time.time()

    # ── Load ──────────────────────────────────────────────
    img_nib      = nib.load(IMAGES_TR / f"{case_id}.nii.gz")
    lbl_nib      = nib.load(LABELS_TR / f"{case_id}.nii.gz")
    img          = img_nib.get_fdata()
    lbl          = lbl_nib.get_fdata()
    orig_spacing = img_nib.header.get_zooms()[:3]

    # ── Resample ──────────────────────────────────────────
    img_res = resample_volume(img, orig_spacing, TARGET_SPACING, is_label=False)
    lbl_res = resample_volume(lbl, orig_spacing, TARGET_SPACING, is_label=True)

    # ── Pad/Crop ──────────────────────────────────────────
    img_pad = pad_or_crop_to_shape(img_res, TARGET_SHAPE, pad_value=0)
    lbl_pad = pad_or_crop_to_shape(lbl_res, TARGET_SHAPE, pad_value=0)

    # ── Normalise ─────────────────────────────────────────
    img_norm, mean, std = zscore_normalise(img_pad)

    # ── Save ──────────────────────────────────────────────
    np.save(PREP_IMGS_TR / f"{case_id}.npy", img_norm.astype(np.float32))
    np.save(PREP_LBLS_TR / f"{case_id}.npy", lbl_pad.astype(np.uint8))

    elapsed = time.time() - start
    norm_stats.append({
        "case_id"  : case_id,
        "mean"     : round(float(mean), 3),
        "std"      : round(float(std), 3),
        "atrium_%" : round(100 * (lbl_pad == 1).sum() / lbl_pad.size, 4)
    })

    print(f"[{idx+1:02d}/20] {case_id} | "
          f"shape={img_norm.shape} | "
          f"mean={mean:.1f} std={std:.1f} | "
          f"{elapsed:.1f}s")

total_time = time.time() - start_total
print(f"\n✅ All 20 training cases done in {total_time:.1f}s")

RE-PREPROCESSING TRAINING CASES (320,320,153)
[01/20] la_003 | shape=(320, 320, 153) | mean=260.2 std=285.0 | 1.5s
[02/20] la_004 | shape=(320, 320, 153) | mean=204.8 std=255.8 | 1.3s
[03/20] la_005 | shape=(320, 320, 153) | mean=233.9 std=253.0 | 1.4s
[04/20] la_007 | shape=(320, 320, 153) | mean=262.6 std=248.6 | 1.6s
[05/20] la_009 | shape=(320, 320, 153) | mean=247.6 std=287.9 | 1.2s
[06/20] la_010 | shape=(320, 320, 153) | mean=243.5 std=276.2 | 1.4s
[07/20] la_011 | shape=(320, 320, 153) | mean=295.7 std=297.1 | 1.4s
[08/20] la_014 | shape=(320, 320, 153) | mean=284.2 std=283.3 | 1.4s
[09/20] la_016 | shape=(320, 320, 153) | mean=292.3 std=300.4 | 1.1s
[10/20] la_017 | shape=(320, 320, 153) | mean=271.1 std=280.1 | 1.4s
[11/20] la_018 | shape=(320, 320, 153) | mean=221.3 std=255.4 | 1.4s
[12/20] la_019 | shape=(320, 320, 153) | mean=245.8 std=263.6 | 1.2s
[13/20] la_020 | shape=(320, 320, 153) | mean=242.9 std=273.1 | 1.3s
[14/20] la_021 | shape=(320, 320, 153) | mean=298.3 std=2